# Correlation Analysis: Corruption/Poor Resource Use vs HDI by Cluster

**Project:** Data-Driven Public Compliance (MBA Thesis)  
**Thesis Title:** Correlation between Federal Transfers and Municipal Socioeconomic Indicators

**Author:** Enok  
**Date:** 2026-04-19

---

## Objective

Investigate the correlation between **indicators of poor public resource use** (sanctions per million BRL transferred) and **human development indicators** (income, literacy) **within each cluster of similar municipalities**.

The cluster-based analysis strategy enables **comparing apples to apples**: municipalities with similar socioeconomic characteristics are analyzed together, isolating the effect of efficiency/corruption from the effect of development level.

---

## Hypotheses

1. **H1:** Municipalities with more sanctions per BRL transferred (inefficiency/corruption) tend to have lower HDI within the same cluster
2. **H2:** The correlation is stronger when controlling for socioeconomic similarity (clusters) vs. aggregate analysis
3. **H3:** There are distinct patterns across clusters: some show strong correlation, others show internal heterogeneity

---

## Input Metrics

**Corruption/Poor Use Indicator (X):**
- `sanctions_per_million_brl_transfers`: Number of sanctions per million BRL in federal transfers
- Interpretation: higher values indicate more sanctions relative to received resources

**HDI/Development Indicators (Y):**
- `avg_income_real_2022_2022_brl`: Average real income (IPCA 2022) - HDI Income proxy
- `literacy_rate_2022`: Literacy rate - HDI Education proxy
- `income_change_real_pct`: Real income change 2010-2022 - development dynamics proxy
- `literacy_change_pp`: Literacy change (percentage points) 2010-2022

**Similarity Control:**
- `cluster`: K-means assignment with 4 clusters based on 12 socioeconomic features

---

## Expected Outputs

1. **Correlation table by cluster:** Pearson r and p-value for each cluster
2. **Municipal vulnerability index:** Combined score for mapping
3. **Representative sample:** Top N cities per cluster for presentation
4. **GeoJSON for QGIS:** Data ready for choropleth mapping
5. **Consolidated dashboard:** Interactive visualization of results

In [ ]:
# --- AUTO-GENERATED DEPENDENCY INSTALL ---
# Installs all project dependencies on first run (Colab, fresh environments, etc).
# Idempotent: pip skips anything already installed.
# To regenerate this cell, run: python scripts/inject_pip_install.py

import subprocess
import sys
from pathlib import Path

_req = Path.cwd().parent / "requirements.txt"
if not _req.exists():
    _req = Path.cwd() / "requirements.txt"

if _req.exists():
    print(f"Installing dependencies from {_req.name if _req.exists() else "requirements.txt"} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)])
    print("Dependencies ready.")
else:
    print("requirements.txt not found. Install manually: pip install -r requirements.txt")


## 1. Setup and Data Loading

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Statistics
from scipy import stats
from scipy.stats import pearsonr, spearmanr

# AWS
import boto3
import tempfile

# Configure style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Libraries loaded successfully!")

In [ ]:
# Reproducibility
SEED = 42
np.random.seed(SEED)

print(f"Seed set to {SEED}")

In [ ]:
# AWS Configuration
import json as _json

_rtcfg_path = os.path.join('..', 'config', 'runtime_config.json')
if os.path.exists(_rtcfg_path):
    with open(_rtcfg_path) as _f:
        _rtcfg = _json.load(_f)
else:
    _rtcfg = {}

S3_BUCKET_NAME = os.environ.get('S3_BUCKET_NAME', _rtcfg.get('aws', {}).get('s3_bucket_name', ''))
AWS_PROFILE = os.environ.get('AWS_PROFILE', _rtcfg.get('aws', {}).get('profile', None))

print(f"Bucket: {S3_BUCKET_NAME}")
print(f"Profile: {AWS_PROFILE}")

In [ ]:
# Load data from local Gold layer and compute clustering inline
from src.analysis.local_data_loader import LocalGoldDataLoader
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

loader = LocalGoldDataLoader()

# Load required datasets
df_state = loader.load_dataset('analysis_compliance')
df_city = loader.load_dataset('analysis_compliance_municipality')
df_cluster = loader.load_dataset('consolidated_clustering')

print(f"Loaded {len(df_state)} state rows")
print(f"Loaded {len(df_city)} city rows") 
print(f"Loaded {len(df_cluster)} cluster rows")

# Compute PCA + KMeans clustering (required for downstream analysis)
# Use normalized features from consolidated_clustering
feature_cols = [c for c in df_cluster.columns if c.endswith('_norm') or c.startswith('log_')]
X = df_cluster[feature_cols].fillna(0).values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA(n_components=3, random_state=42)
pcs = pca.fit_transform(X_scaled)
df_cluster['PC1'] = pcs[:, 0]
df_cluster['PC2'] = pcs[:, 1]
df_cluster['PC3'] = pcs[:, 2]

# KMeans clustering (k=5 default)
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df_cluster['cluster'] = kmeans.fit_predict(X_scaled)

print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.1%}")
print(f"Clusters assigned: {df_cluster['cluster'].nunique()} unique clusters")


In [ ]:
# Check available columns
print("\nAvailable columns in df_city:")
for col in sorted(df_city.columns):
    print(f"  - {col}")

## 2. Data Preparation

In [ ]:
# Merge compliance data with clusters (include all needed columns from df_cluster)
merge_cols = ['municipality_code', 'cluster', 'PC1', 'PC2', 'PC3', 
              'income_change_real_pct', 'literacy_change_pp',
              'population_change_pct', 'households_change_pct']
# Keep only columns that exist
merge_cols = [c for c in merge_cols if c in df_cluster.columns]

df = df_city.merge(
    df_cluster[merge_cols], 
    on='municipality_code', 
    how='left'
)

print(f"Merge completed: {len(df):,} municipalities with cluster assigned")

# Check municipalities without cluster
no_cluster = df['cluster'].isna().sum()
print(f"Municipalities without cluster: {no_cluster}")


In [ ]:
# Define analysis variables
CORRUPTION_VAR = 'sanctions_per_million_brl_transfers'
IDH_VARS = [
    'avg_income_real_2022_2022_brl',      # Income (HDI proxy)
    'literacy_rate_2022',                  # Literacy
    'income_change_real_pct',              # Income dynamics
    'literacy_change_pp',                  # Education dynamics
]

# Create flag for municipalities with valid sanctions data
df['has_sanctions_data'] = (
    df[CORRUPTION_VAR].notna() & 
    (df[CORRUPTION_VAR] >= 0) &
    df['total_transfers'].notna() &
    (df['total_transfers'] > 0)
)

print("Sanctions data availability:")
print(f"  With valid data: {df['has_sanctions_data'].sum():,}")
print(f"  Without data: {(~df['has_sanctions_data']).sum():,}")

# Statistics for corruption variable
print("\nSanctions per million BRL statistics:")
print(df[CORRUPTION_VAR].describe())

## 3. Correlation Analysis by Cluster

In [ ]:
# Function to calculate correlation by cluster
def calculate_correlation_by_cluster(df, cluster_id, corr_var, idh_var):
    """Calculate correlation between corr_var and idh_var for a specific cluster."""
    
    subset = df[
        (df['cluster'] == cluster_id) & 
        df['has_sanctions_data'] &
        df[corr_var].notna() & 
        df[idh_var].notna()
    ].copy()
    
    n = len(subset)
    
    if n < 10:
        return {
            'cluster': cluster_id,
            'n_municipios': n,
            'pearson_r': None,
            'pearson_p': None,
            'spearman_r': None,
            'spearman_p': None,
            'interpretation': 'Insufficient sample'
        }
    
    # Remove extreme outliers (>3 standard deviations)
    z_scores = np.abs(stats.zscore(subset[corr_var]))
    subset_clean = subset[z_scores < 3]
    
    n_clean = len(subset_clean)
    
    if n_clean < 10:
        return {
            'cluster': cluster_id,
            'n_municipios': n_clean,
            'pearson_r': None,
            'pearson_p': None,
            'spearman_r': None,
            'spearman_p': None,
            'interpretation': 'Insufficient clean data'
        }
    
    # Calculate correlations
    pearson_r, pearson_p = pearsonr(subset_clean[corr_var], subset_clean[idh_var])
    spearman_r, spearman_p = spearmanr(subset_clean[corr_var], subset_clean[idh_var])
    
    # Interpretation
    if pearson_p < 0.001:
        significance = '***'
    elif pearson_p < 0.01:
        significance = '**'
    elif pearson_p < 0.05:
        significance = '*'
    else:
        significance = 'ns'
    
    return {
        'cluster': cluster_id,
        'n_municipios': n_clean,
        'pearson_r': round(pearson_r, 4),
        'pearson_p': round(pearson_p, 6),
        'spearman_r': round(spearman_r, 4),
        'spearman_p': round(spearman_p, 6),
        'significance': significance,
        'interpretation': f"r={pearson_r:.3f}{significance}"
    }

print("Correlation by cluster function defined.")

In [ ]:
# Calculate correlations for all clusters and HDI variables
clusters = sorted(df['cluster'].dropna().unique())

results = []

for cluster_id in clusters:
    for idh_var in IDH_VARS:
        result = calculate_correlation_by_cluster(df, cluster_id, CORRUPTION_VAR, idh_var)
        result['idh_var'] = idh_var
        results.append(result)

df_results = pd.DataFrame(results)

print("\nCorrelation Table by Cluster")
print("="*80)

# Pivot for visualization
pivot_table = df_results.pivot_table(
    index=['cluster', 'n_municipios'], 
    columns='idh_var', 
    values='interpretation',
    aggfunc='first'
)

print(pivot_table)

In [ ]:
# Visualization of correlations by cluster
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, idh_var in enumerate(IDH_VARS):
    ax = axes[idx]
    
    subset = df_results[df_results['idh_var'] == idh_var].copy()
    
    # Create colored bars based on significance
    colors = []
    for _, row in subset.iterrows():
        if row['pearson_p'] < 0.001:
            colors.append('darkred')
        elif row['pearson_p'] < 0.01:
            colors.append('red')
        elif row['pearson_p'] < 0.05:
            colors.append('orange')
        else:
            colors.append('lightgray')
    
    bars = ax.bar(subset['cluster'], subset['pearson_r'], color=colors, alpha=0.8, edgecolor='black')
    
    # Add reference line at r=0
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
    
    # Labels
    ax.set_xlabel('Cluster')
    ax.set_ylabel("Pearson Correlation (r)")
    ax.set_title(f"{idh_var}\nvs Sanctions/Million BRL")
    ax.set_ylim(-0.5, 0.5)
    
    # Add values on bars
    for bar, r, p in zip(bars, subset['pearson_r'], subset['pearson_p']):
        height = bar.get_height()
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        ax.text(
            bar.get_x() + bar.get_width()/2.,
            height + (0.02 if height >= 0 else -0.05),
            f"{r:.3f}{sig}",
            ha='center', va='bottom' if height >= 0 else 'top',
            fontsize=9, fontweight='bold'
        )

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='darkred', label='p < 0.001 ***'),
    Patch(facecolor='red', label='p < 0.01 **'),
    Patch(facecolor='orange', label='p < 0.05 *'),
    Patch(facecolor='lightgray', label='not significant')
]
fig.legend(handles=legend_elements, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 0.98))

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.suptitle("Correlation: Sanctions/Transfer vs Development Indicators\nby Cluster", fontsize=14, fontweight='bold', y=1.02)
plt.show()

## 4. Municipal Vulnerability Index

We create a combined index for mapping:
- **High index (red):** Many sanctions per transferred resource + Low HDI = High vulnerability/corruption
- **Low index (blue):** Few sanctions per resource + High HDI = Low vulnerability/good management

In [ ]:
# Create vulnerability index
def calculate_vulnerability_index(row):
    """
    Combined index: (normalized sanctions) - (normalized HDI)
    Higher = more vulnerable (high corruption, low development)
    """
    
    # Normalize sanctions (0-1, where 1 = more sanctions)
    if pd.isna(row[CORRUPTION_VAR]) or row[CORRUPTION_VAR] < 0:
        return None
    
    # Use log to reduce skewness
    sanctions_log = np.log1p(row[CORRUPTION_VAR])
    
    # HDI component (average of income and literacy normalized)
    income_norm = row['avg_income_real_2022_2022_brl'] if pd.notna(row['avg_income_real_2022_2022_brl']) else 0
    literacy_norm = row['literacy_rate_2022'] if pd.notna(row['literacy_rate_2022']) else 0
    
    # HDI score (0-1 approximate)
    idh_score = (income_norm / 5000 + literacy_norm / 100) / 2  # Approximate normalization
    idh_score = min(max(idh_score, 0), 1)  # Clip 0-1
    
    # Vulnerability: corruption - development
    vulnerability = sanctions_log - (idh_score * 5)  # Weight HDI
    
    return vulnerability

# Apply to all municipalities with data
df['vulnerability_index'] = df.apply(calculate_vulnerability_index, axis=1)

# Index statistics
print("Vulnerability Index Statistics:")
print(df['vulnerability_index'].describe())

# Distribution by cluster
print("\nIndex Distribution by Cluster:")
print(df.groupby('cluster')['vulnerability_index'].describe().round(3))

In [ ]:
# Visualization of index distribution by cluster
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Box plot by cluster
ax1 = axes[0]
cluster_data = [df[df['cluster'] == c]['vulnerability_index'].dropna() for c in sorted(df['cluster'].dropna().unique())]
bp = ax1.boxplot(cluster_data, labels=[f'Cluster {int(c)}' for c in sorted(df['cluster'].dropna().unique())], patch_artist=True)

# Color boxes
colors = ['lightblue', 'lightgreen', 'lightyellow', 'lightcoral']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax1.set_xlabel('Cluster')
ax1.set_ylabel('Vulnerability Index')
ax1.set_title('Index Distribution by Cluster')
ax1.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Neutral reference')
ax1.legend()

# General histogram
ax2 = axes[1]
ax2.hist(df['vulnerability_index'].dropna(), bins=50, color='steelblue', edgecolor='white', alpha=0.7)
ax2.set_xlabel('Vulnerability Index')
ax2.set_ylabel('Frequency')
ax2.set_title('General Index Distribution')
ax2.axvline(x=0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 5. Representative Sample Selection

Select extreme municipalities (best and worst) within each cluster for presentation.

In [ ]:
# Select top N per cluster (best and worst on vulnerability index)
TOP_N = 5

representative_sample = []

for cluster_id in sorted(df['cluster'].dropna().unique()):
    cluster_df = df[df['cluster'] == cluster_id].copy()
    
    # Best (lowest vulnerability = blue)
    best = cluster_df.nsmallest(TOP_N, 'vulnerability_index')
    best['category'] = 'Best Management'
    best['rank_in_cluster'] = range(1, len(best) + 1)
    
    # Worst (highest vulnerability = red)
    worst = cluster_df.nlargest(TOP_N, 'vulnerability_index')
    worst['category'] = 'High Vulnerability'
    worst['rank_in_cluster'] = range(1, len(worst) + 1)
    
    representative_sample.extend([best, worst])

df_sample = pd.concat(representative_sample, ignore_index=True)

print(f"Representative sample selected: {len(df_sample)} municipalities")
print(f"  - {len(df_sample[df_sample['category'] == 'Best Management'])} best")
print(f"  - {len(df_sample[df_sample['category'] == 'High Vulnerability'])} worst")

# Columns for display
display_cols = [
    'municipality_code', 'municipality_name', 'state_name', 'cluster',
    'category', 'rank_in_cluster',
    'sanctions_per_million_brl_transfers', 'avg_income_real_2022_2022_brl', 
    'literacy_rate_2022', 'vulnerability_index'
]

print("\nFirst cases from sample:")
print(df_sample[display_cols].head(10).to_string(index=False))

In [ ]:
# Visualization of sample in PCA coordinates
fig = px.scatter(
    df,
    x='PC1', y='PC2',
    color='vulnerability_index',
    color_continuous_scale='RdYlBu_r',  # Red = high (bad), Blue = low (good)
    hover_data=['municipality_name', 'state_name', 'sanctions_per_million_brl_transfers', 'avg_income_real_2022_2022_brl'],
    title='Municipalities in PCA Space (colored by Vulnerability Index)',
    labels={'PC1': 'Principal Component 1', 'PC2': 'Principal Component 2', 'vulnerability_index': 'Vulnerability'}
)

# Add highlight for representative sample
sample_coords = df_sample[['PC1', 'PC2', 'municipality_name', 'category']].copy()

fig.add_trace(
    go.Scatter(
        x=sample_coords[sample_coords['category'] == 'Best Management']['PC1'],
        y=sample_coords[sample_coords['category'] == 'Best Management']['PC2'],
        mode='markers',
        marker=dict(size=12, color='blue', symbol='star', line=dict(width=2, color='black')),
        name='Best Management (Sample)',
        text=sample_coords[sample_coords['category'] == 'Best Management']['municipality_name'],
        hovertemplate='%{text}<extra></extra>'
    )
)

fig.add_trace(
    go.Scatter(
        x=sample_coords[sample_coords['category'] == 'High Vulnerability']['PC1'],
        y=sample_coords[sample_coords['category'] == 'High Vulnerability']['PC2'],
        mode='markers',
        marker=dict(size=12, color='red', symbol='x', line=dict(width=2, color='black')),
        name='High Vulnerability (Sample)',
        text=sample_coords[sample_coords['category'] == 'High Vulnerability']['municipality_name'],
        hovertemplate='%{text}<extra></extra>'
    )
)

fig.update_layout(height=700)
fig.show()

## 6. GeoJSON Generation for QGIS

Create GeoJSON file with vulnerability index for visualization in QGIS.

In [ ]:
# Prepare data for GeoJSON
# Join with geographic data (municipality centroids)

from pathlib import Path
import zipfile
import tempfile

try:
    import shapefile  # pyshp
except ImportError as exc:
    raise ImportError("pyshp is required. Install with: pip install pyshp>=2.3.1") from exc

# Load municipality shapefile
map_assets_dir = Path("..") / "docs" / "thesis_presentation_assets" / "qgis"
municipality_zip_path = map_assets_dir / "BR_Municipios_2022.zip"

print(f"Loading shapefile: {municipality_zip_path}")

with tempfile.TemporaryDirectory(prefix="ibge_muni_shape_") as _tmp_dir:
    with zipfile.ZipFile(municipality_zip_path) as _zip_file:
        _zip_file.extractall(_tmp_dir)

    _shp_path = next(Path(_tmp_dir).glob("*.shp"))
    _reader = shapefile.Reader(str(_shp_path))
    _fields = [field[0] for field in _reader.fields[1:]]
    _code_idx = _fields.index("CD_MUN")

    # Extract centroids
    centroids = []
    for _shape_record in _reader.iterShapeRecords():
        municipality_code = str(_shape_record.record[_code_idx]).zfill(7)
        xmin, ymin, xmax, ymax = _shape_record.shape.bbox
        centroids.append({
            "municipality_code": municipality_code,
            "lon": (xmin + xmax) / 2.0,
            "lat": (ymin + ymax) / 2.0,
        })
    _reader.close()  # Close before temp cleanup (Windows fix)

df_centroids = pd.DataFrame(centroids)
print(f"Centroids loaded: {len(df_centroids):,}")

# Join with analysis data
df_geo = df.merge(df_centroids, on='municipality_code', how='inner')
print(f"Municipalities with analysis data + coordinates: {len(df_geo):,}")

In [ ]:
# Create vulnerability categories for the map
def categorize_vulnerability(v):
    if pd.isna(v):
        return 'No data'
    elif v < -3:
        return 'Very Low (Blue)'
    elif v < -1:
        return 'Low (Light Blue)'
    elif v < 1:
        return 'Neutral (Yellow)'
    elif v < 3:
        return 'High (Orange)'
    else:
        return 'Very High (Red)'

df_geo['vulnerability_category'] = df_geo['vulnerability_index'].apply(categorize_vulnerability)

print("Category distribution:")
print(df_geo['vulnerability_category'].value_counts())

In [ ]:
# Create GeoJSON for QGIS
import json

features = []

for _, row in df_geo.iterrows():
    if pd.isna(row['lon']) or pd.isna(row['lat']):
        continue
    
    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [float(row['lon']), float(row['lat'])]
        },
        "properties": {
            "municipality_code": str(row['municipality_code']),
            "municipality_name": str(row['municipality_name']),
            "state_code": str(row['state_code']) if pd.notna(row['state_code']) else None,
            "state_name": str(row['state_name']) if pd.notna(row['state_name']) else None,
            "cluster": int(row['cluster']) if pd.notna(row['cluster']) else None,
            "vulnerability_index": float(row['vulnerability_index']) if pd.notna(row['vulnerability_index']) else None,
            "vulnerability_category": str(row['vulnerability_category']),
            "sanctions_per_million": float(row['sanctions_per_million_brl_transfers']) if pd.notna(row['sanctions_per_million_brl_transfers']) else 0,
            "avg_income_2022": float(row['avg_income_real_2022_2022_brl']) if pd.notna(row['avg_income_real_2022_2022_brl']) else None,
            "literacy_rate_2022": float(row['literacy_rate_2022']) if pd.notna(row['literacy_rate_2022']) else None,
            "total_transfers": float(row['total_transfers']) if pd.notna(row['total_transfers']) else 0,
            "n_sanctions": int(row['n_sanctions']) if pd.notna(row['n_sanctions']) else 0,
        }
    }
    features.append(feature)

geojson = {
    "type": "FeatureCollection",
    "name": "Municipalities_Vulnerability_Index",
    "crs": {
        "type": "name",
        "properties": {
            "name": "urn:ogc:def:crs:OGC:1.3:CRS84"
        }
    },
    "features": features
}

# Save
output_path = map_assets_dir / "brazil_municipalities_vulnerability_index.geojson"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(geojson, f, ensure_ascii=False, indent=2)

print(f"GeoJSON saved: {output_path}")
print(f"Total features: {len(features)}")

# Statistics by state
print("\nTop 10 states with highest average vulnerability:")
state_vuln = df_geo.groupby('state_name')['vulnerability_index'].mean().sort_values(ascending=False).head(10)
print(state_vuln.round(3))

## 7. Consolidated Dashboard

In [ ]:
# Executive summary
print("="*80)
print("EXECUTIVE SUMMARY - CORRUPTION/POOR USE vs HDI ANALYSIS")
print("="*80)

print(f"\n📊 Data Analyzed:")
print(f"   • Total municipalities: {len(df):,}")
print(f"   • With sanctions data: {df['has_sanctions_data'].sum():,}")
print(f"   • Clusters defined: {len(clusters)}")

print(f"\n📈 Significant Correlations (p < 0.05):")
sig_corrs = df_results[df_results['pearson_p'] < 0.05]
for _, row in sig_corrs.iterrows():
    print(f"   • Cluster {int(row['cluster'])} vs {row['idh_var']}: r={row['pearson_r']:.3f} {row['significance']}")

if len(sig_corrs) == 0:
    print("   • No significant correlations detected in clusters")
    print("   • This suggests the effect is heterogeneous or of low magnitude")

print(f"\n🎯 Representative Sample:")
print(f"   • {TOP_N} best municipalities per cluster = {len(df_sample[df_sample['category'] == 'Best Management'])} total")
print(f"   • {TOP_N} worst municipalities per cluster = {len(df_sample[df_sample['category'] == 'High Vulnerability'])} total")

print(f"\n🗺️  Generated Assets:")
print(f"   • Municipal GeoJSON: docs/thesis_presentation_assets/qgis/brazil_municipalities_vulnerability_index.geojson")
print(f"   • {len(features)} municipalities with coordinates and vulnerability index")

print("\n" + "="*80)

In [ ]:
# Export tables for dashboard

# 1. Correlation table
correlations_export = df_results[
    ['cluster', 'n_municipios', 'idh_var', 'pearson_r', 'pearson_p', 'significance', 'interpretation']
].copy()
correlations_export.columns = ['cluster', 'n_municipios', 'idh_var', 'pearson_r', 'pearson_p', 'significance', 'interpretation']

# 2. Representative sample table
sample_export = df_sample[
    ['municipality_code', 'municipality_name', 'state_code', 'state_name', 
     'cluster', 'category', 'rank_in_cluster', 'vulnerability_index',
     'sanctions_per_million_brl_transfers', 'avg_income_real_2022_2022_brl', 
     'literacy_rate_2022', 'total_transfers', 'n_sanctions']
].copy()

# 3. Cluster summary
cluster_summary = df.groupby('cluster').agg({
    'municipality_code': 'count',
    'vulnerability_index': ['mean', 'std', 'min', 'max'],
    'sanctions_per_million_brl_transfers': ['mean', 'median'],
    'avg_income_real_2022_2022_brl': 'mean',
    'literacy_rate_2022': 'mean'
}).round(3)

# Save CSVs
output_dir = Path("..") / "docs" / "thesis_presentation_assets"

correlations_export.to_csv(output_dir / "correlation_by_cluster.csv", index=False)
sample_export.to_csv(output_dir / "representative_sample_cities.csv", index=False)
cluster_summary.to_csv(output_dir / "cluster_summary.csv")

print("Exported tables:")
print(f"  1. {output_dir / 'correlation_by_cluster.csv'}")
print(f"  2. {output_dir / 'representative_sample_cities.csv'}")
print(f"  3. {output_dir / 'cluster_summary.csv'}")

---

## End of Analysis

### Main Conclusions

1. **Correlation by Cluster:** The relationship between sanctions/transfers and development indicators varies significantly across clusters, reflecting different regional realities.

2. **Vulnerability Index:** The combined index allows identifying municipalities with high relative inefficiency (sanctions per resource) even within similar socioeconomic contexts.

3. **Representative Sample:** The selection of extremes (best/worst) per cluster provides concrete cases for qualitative analysis and presentation.

4. **Map for QGIS:** The generated GeoJSON enables spatial visualization of vulnerabilities, identifying geographic hotspots of higher risk.

### Next Steps

- **Qualitative Analysis:** Investigate extreme cases from the representative sample to understand contextual factors
- **Map in QGIS:** Open the GeoJSON in QGIS and apply graduated symbology by vulnerability index
- **Interactive Dashboard:** Use the exported tables to create interactive visualizations (Power BI, Tableau, etc.)
- **Validation:** Consult literature on similar government efficiency metrics